# 10. Импутация пропусков

Рабочий ноутбук к главе книги `../chapters/10-imputation.md`: здесь только задания и код, теория и выводы — в главе. Источник — глава 15 пособия.

Графики для главы сохраняй через `save_fig(fig, "имя")` → `../chapters/img/10-имя.png`.

In [ ]:
import logging
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from etna.datasets import TSDataset, generate_periodic_df
from etna.transforms import TimeSeriesImputerTransform
from etna.models import SeasonalMovingAverageModel
from etna.pipeline import Pipeline
from etna.metrics import SMAPE, MAE
from etna.analysis import plot_imputation, plot_backtest

logging.getLogger("cmdstanpy").disabled = True
warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = "../local/data"
IMG_DIR = "../chapters/img"  # графики для главы книги
HORIZON = 14
RANDOM_SEED = 42


def save_fig(fig, name: str) -> None:
    """Сохранить график для главы книги: ../chapters/img/10-<name>.png."""
    os.makedirs(IMG_DIR, exist_ok=True)
    fig.savefig(f"{IMG_DIR}/10-{name}.png", dpi=150, bbox_inches="tight")

## Задание 1. Синтетика: стратегии заполнения

Ряд с недельной сезонностью, где мы знаем правильные значения на месте пропусков.

### Шаг 1.1. Ряд с пропусками

Синтетический ряд с недельной сезонностью; убери случайные ~10% точек и один блок из 7 дней подряд, сохрани исходные значения.

### Шаг 1.2. Все стратегии

Заполни пропуски всеми стратегиями `TimeSeriesImputerTransform`: `constant`, `mean`, `running_mean`, `forward_fill`, `seasonal`, `seasonal_nonautoreg`; нарисуй `plot_imputation`.

### Шаг 1.3. Ошибка заполнения

MAE заполненных значений против исходных — отдельно для одиночных пропусков и для блока из 7 дней.

## Задание 2. Реальные юниты: где у нас пропуски

10 юнитов с 2025-01-01.

### Шаг 2.1. Загрузка

Прочитай `facts-by-date.csv` и `units.csv` из `DATA_DIR`, собери `TSDataset` (дневная частота, `segment` = `MaskedUnitId`). Период — с 2025-01-01.

### Шаг 2.2. Пропуски и нули

Найди пропуски и нулевые дни по юнитам (`ts.describe()`); у каких юнитов нули — это «юнит ещё не открылся»?

## Задание 3. Импутация и прогноз

Влияет ли способ заполнения на качество прогноза.

### Шаг 3.1. Искусственные пропуски

На 2 юнитах `10+` (с 2026-04-01) убери ~10% дней train.

### Шаг 3.2. Backtest стратегий

`Pipeline(SeasonalMovingAverageModel(window=4, seasonality=7), transforms=[TimeSeriesImputerTransform(strategy=...)])` — backtest для 3–4 стратегий и для ряда без пропусков.